# Advanced functionalities

In the example notebook, we learned how to measure the grouping loss and grouping risks with partitioning estimates fitted on the residuals. We can also use broader models such as **Histogram Gradient Boosting** or **Random Forests** directly fitted on the residuals to provide estimates. 


> **⚠️ Warning:** These estimates come with no theoretical guarantee in our work.

In [1]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=200000, random_state=42, n_features=20, n_informative=20, n_redundant=0
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=0)

est = LogisticRegression()
est.fit(X_train, y_train)
S_test = est.predict_proba(X_test)[:, 1]

To indicate that we aren't using a partitioning estimator, we use the `residual_estimator` option.

In [2]:
from glest.core import GLEstimator
from sklearn.ensemble import RandomForestRegressor


gle_custom = GLEstimator(
    residual_estimator=RandomForestRegressor(n_estimators=100, random_state=42)
)
gle_custom.fit(X_test, S_test, y_test)
gle_custom.estimate()

print(gle_custom)

GLEstimator()
  Scoring Rule      : Brier: 0.1403
  Grouping loss     : 0.1313
  Calibration Loss  : 0.0025
  Epistemic Loss    : 0.1338



The same goes for the RiskEstimator

In [3]:
from glest.core import RiskEstimator

X_test, X_risk, y_test, y_risk = train_test_split(
    X_test, y_test, test_size=0.5, random_state=0
)


S_test = est.predict_proba(X_test)[:, 1]

risk = RiskEstimator(
    residual_estimator=RandomForestRegressor(n_estimators=100, random_state=42)
)
risk.fit(X_test, S_test, y_test)
risk.predict(X_risk, est.predict_proba(X_risk)[:, 1].reshape(-1, 1), t=0.5)

(array([0., 0., 0., ..., 0., 0., 0.], shape=(50000,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(50000,)))